In [1]:
%pip install --upgrade --quiet google-genai pandas

Note: you may need to restart the kernel to use updated packages.


# Japan Travel Multimodal Chat Assistant

Features:
- Function Calling (Weather)
- Multimodal Input (Image/PDF)
- Conversational Memory
- Controlled JSON Output
- Safety Settings
- JSON Visualization


### Imports

In [2]:
import os
import json
import pandas as pd
from IPython.display import display , HTML
from typing import List, Dict, Any
import matplotlib.pyplot as plt
from google import genai
from google.genai.types import (
    FunctionDeclaration,
    GenerateContentConfig,
    Part,
    Tool,
    SafetySetting,
    HarmCategory,
    HarmBlockThreshold,
)


### Configuration

In [8]:
PROJECT_ID = "qwiklabs-gcp-03-2793f8f63220"
LOCATION = os.environ.get("GOOGLE_CLOUD_REGION", "us-central1")
MODEL_ID = "gemini-2.5-flash"

client = genai.Client(
    vertexai=True,
    project=PROJECT_ID,
    location=LOCATION,
)


### Dummy Weather Function


In [9]:
class WeatherService:

    def __init__(self):
        self.data = {
            "Tokyo": {"temperature_celsius": 24, "description": "Partly cloudy"},
            "Kyoto": {"temperature_celsius": 22, "description": "Clear sky"},
        }

    def get_weather(self, city: str) -> Dict[str, Any]:
        return {
            "city": city,
            **self.data.get(city, {"temperature_celsius": 20, "description": "Unknown"})
        }


## Memory

In [10]:
class ChatMemory:

    def __init__(self):
        self.history: List[Any] = []

    def add_user(self, message):
        if isinstance(message, list):
            self.history.extend(message)
        else:
            self.history.append(message)

    def add_model(self, text):
        self.history.append(text)

    def add_tool(self, function_name, response):
        self.history.append(
            Part.from_function_response(
                name=function_name,
                response=response,
            )
        )

    def get(self):
        return self.history


### Main Assistant

In [11]:
class TravelChatAssistant:

    def __init__(self):
        self.weather_service = WeatherService()
        self.memory = ChatMemory()
        self.tool = self._build_tool()

    def _build_tool(self):
        weather_function = FunctionDeclaration(
            name="get_weather",
            description="Retrieve weather information",
            parameters={
                "type": "OBJECT",
                "properties": {
                    "city": {"type": "STRING"}
                },
                "required": ["city"],
            },
        )
        return Tool(function_declarations=[weather_function])

    def _system_instruction(self):
        return """
    คุณคือ Japan Travel Expert ระดับไกด์ท้องถิ่นมืออาชีพ
    มีประสบการณ์มากกว่า 15 ปี เชี่ยวชาญเรื่อง:
    - สภาพอากาศญี่ปุ่นตามฤดูกาล
    - ภูมิประเทศแต่ละภูมิภาค
    - ผลกระทบของอากาศต่อการท่องเที่ยว
    - พฤติกรรมนักท่องเที่ยวในแต่ละช่วงเวลา
    - ร้านอาหาร local ที่คนญี่ปุ่นนิยมจริง

    หน้าที่ของคุณ:
    วิเคราะห์อย่างละเอียดก่อนวางแผนทุกครั้ง
    และต้องเรียกใช้ function get_weather ก่อนจัด itinerary

    วิเคราะห์เชิงลึกในประเด็นต่อไปนี้:
    1. สภาพอากาศ (อุณหภูมิ / ฝน / หิมะ / ความชื้น / ลมแรง)
    2. ความเสี่ยงตามฤดูกาล (ไต้ฝุ่น / หิมะตกหนัก / ใบไม้ยังไม่เปลี่ยนสี)
    3. เสื้อผ้าที่ควรเตรียม
    4. ข้อดีข้อเสียของการเที่ยวช่วงนั้น
    5. จุดชมธรรมชาติที่เหมาะกับฤดูกาล
    6. ร้านอาหารตามฤดูกาล
    7. พาสเดินทางที่คุ้มค่า
    8. เทคนิคเลี่ยงคนเยอะ
    9. คำเตือนที่นักท่องเที่ยวมักไม่รู้

    IMPORTANT RULES:
    1. MUST call get_weather function before planning.
    2. MUST maintain conversation context.
    3. MUST respond ONLY in valid JSON format.
    4. DO NOT include any explanation outside JSON.
    5. Ensure the JSON structure strictly follows the schema below.

    REQUIRED JSON SCHEMA:

    {
      "trip_plan": {
        "city": "string",
        "duration_days": number,
        "current_weather": {
          "description": "string",
          "temperature_celsius": number
        },
        "weather_analysis": {
          "seasonal_risks": "string",
          "recommended_clothing": "string",
          "travel_impact_summary": "string"
        },
        "pros_cons": {
          "advantages": [],
          "disadvantages": []
        },
        "itinerary": [
          {
            "day": number,
            "theme": "string",
            "activities": {
              "morning": "string",
              "afternoon": "string",
              "evening": "string"
            },
            "food_suggestions": []
          }
        ],
        "seasonal_food": [],
        "transport_pass_recommendation": "string",
        "crowd_avoidance_tips": [],
        "important_warnings": []
      }
    }
    """


    def send_message(self, user_input, file_uri=None, mime_type=None):

        # ----- Multimodal Support -----
        if file_uri:
            user_parts = [
                user_input,
                Part.from_uri(file_uri=file_uri, mime_type=mime_type)
            ]
        else:
            user_parts = user_input

        self.memory.add_user(user_parts)

        response = client.models.generate_content(
            model=MODEL_ID,
            contents=self.memory.get(),
            config=GenerateContentConfig(
                system_instruction=self._system_instruction(),
                tools=[self.tool],
                temperature=0.3,
                response_mime_type="application/json",
                safety_settings=[
                    SafetySetting(
                        category=HarmCategory.HARM_CATEGORY_DANGEROUS_CONTENT,
                        threshold=HarmBlockThreshold.BLOCK_MEDIUM_AND_ABOVE,
                    )
                ],
            ),
        )

        part = response.candidates[0].content.parts[0]

        # ----- Function Calling -----
        if hasattr(part, "function_call") and part.function_call:

            function_call = part.function_call
            args = function_call.args

            weather_data = self.weather_service.get_weather(args["city"])

            self.memory.add_model(response.candidates[0].content)
            self.memory.add_tool(function_call.name, weather_data)

            final_response = client.models.generate_content(
                model=MODEL_ID,
                contents=self.memory.get(),
                config=GenerateContentConfig(
                    temperature=0.3,
                    response_mime_type="application/json",
                ),
            )

            self.memory.add_model(final_response.candidates[0].content)

            return final_response.text

        else:
            self.memory.add_model(response.candidates[0].content)
            return response.text


## Run Assistant

In [12]:
# สร้าง assistant
assistant = TravelChatAssistant()

# เรียกใช้งาน
result_json = assistant.send_message("Plan a detailed 3-day trip to Tokyo in spring.")

# แสดง raw JSON
print("Raw JSON Output:\n")
print(result_json)


Raw JSON Output:

{
  "trip_name": "3-Day Tokyo Spring Adventure",
  "destination": "Tokyo, Japan",
  "duration_days": 3,
  "season": "Spring",
  "weather_info": {
    "city": "Tokyo",
    "description": "Partly cloudy",
    "temperature_celsius": 24
  },
  "daily_plan": [
    {
      "day": "Day 1",
      "theme": "Historic Charm & Cherry Blossom Serenity",
      "activities": [
        {
          "time": "Morning",
          "description": "Arrive in Tokyo, check into your accommodation. Head to Ueno Park, one of Tokyo's most popular cherry blossom viewing spots. Explore the park, visit a museum (e.g., Tokyo National Museum or Tokyo Metropolitan Art Museum) if time permits.",
          "location": "Ueno Park, Ueno",
          "notes": "Wear comfortable shoes. Best for cherry blossom viewing in late March to early April. Expect crowds."
        },
        {
          "time": "Afternoon",
          "description": "Visit Senso-ji Temple, Tokyo's oldest temple, and stroll through Nakami

## 📊 JSON Visualization
This section visualizes the itinerary in a simple and readable format.


In [15]:
import json
from IPython.display import display, HTML

def visualize_trip_web(json_string):

    data = json.loads(json_string)

    trip_name = data["trip_name"]
    destination = data["destination"]
    duration = data["duration_days"]
    season = data["season"]

    weather = data["weather_info"]

    html = f"""
    <style>
        body {{
            background-color: #f8f9fa;
            font-family: 'Segoe UI', sans-serif;
        }}
        .container {{
            max-width: 1100px;
            margin: auto;
            padding: 20px;
        }}
        .hero {{
            background: white;
            padding: 30px;
            border-radius: 12px;
            box-shadow: 0 4px 15px rgba(0,0,0,0.08);
            margin-bottom: 25px;
        }}
        .hero h1 {{
            margin: 0;
            font-size: 28px;
        }}
        .subtitle {{
            color: #333;
            margin-top: 10px;
        }}
        .card {{
            background: white;
            padding: 20px;
            margin-bottom: 20px;
            border-radius: 12px;
            box-shadow: 0 4px 15px rgba(0,0,0,0.06);
        }}
        .day-title {{
            font-size: 20px;
            font-weight: bold;
            margin-bottom: 10px;
        }}
        table {{
            width: 100%;
            border-collapse: collapse;
        }}
        th, td {{
            border-bottom: 1px solid #ddd;
            padding: 8px;
            text-align: left;
        }}
        th {{
            background: #f4f4f4;
        }}
        ul {{
            margin: 0;
            padding-left: 18px;
        }}
    </style>

    <div class="container">

        <div class="hero">
            <h1>{trip_name}</h1>
            <div class="subtitle">
                Destination: {destination} <br>
                Duration: {duration} days <br>
                Season: {season} <br><br>
                Weather: {weather['description']} | {weather['temperature_celsius']} °C
            </div>
        </div>
    """

    # ---------------------
    # Daily Plan
    # ---------------------
    for day in data["daily_plan"]:
        html += f"""
        <div class="card">
            <div class="day-title">{day['day']} — {day['theme']}</div>

            <table>
                <tr>
                    <th>Time</th>
                    <th>Location</th>
                    <th>Description</th>
                    <th>Notes</th>
                </tr>
        """

        for act in day["activities"]:
            html += f"""
                <tr>
                    <td>{act['time']}</td>
                    <td>{act['location']}</td>
                    <td>{act['description']}</td>
                    <td>{act['notes']}</td>
                </tr>
            """

        html += "</table><br><strong>Meals:</strong><ul>"

        for meal in day["meals"]:
            html += f"<li>{meal['type']}: {meal['suggestion']} ({meal['location_type']})</li>"

        html += "</ul></div>"

    html += "</div>"

    display(HTML(html))


In [16]:
visualize_trip_web(result_json)


Time,Location,Description,Notes
Morning,"Ueno Park, Ueno","Arrive in Tokyo, check into your accommodation. Head to Ueno Park, one of Tokyo's most popular cherry blossom viewing spots. Explore the park, visit a museum (e.g., Tokyo National Museum or Tokyo Metropolitan Art Museum) if time permits.",Wear comfortable shoes. Best for cherry blossom viewing in late March to early April. Expect crowds.
Afternoon,"Senso-ji Temple, Asakusa","Visit Senso-ji Temple, Tokyo's oldest temple, and stroll through Nakamise-dori market street leading up to it. Experience traditional Japanese culture.",Try some local street snacks on Nakamise-dori.
Evening,"Tokyo Skytree, Sumida",Ascend the Tokyo Skytree for panoramic night views of the city. Enjoy the illuminated cityscape.,"Book tickets in advance online to avoid long queues, especially during peak season."
Time,Location,Description,Notes
Morning,"Shibuya Crossing, Shibuya","Experience the iconic Shibuya Crossing, visit the Hachiko statue, and explore the trendy shops and department stores in Shibuya.","Find a good vantage point (e.g., Starbucks Tsutaya) to watch the crossing."
Afternoon,"Takeshita Street & Meiji Jingu Shrine, Harajuku","Immerse yourself in Harajuku's vibrant youth culture at Takeshita Street. Afterwards, find tranquility at the Meiji Jingu Shrine, a peaceful oasis dedicated to Emperor Meiji and Empress Shoken.",Takeshita Street can be very crowded. Be respectful when visiting the shrine.
Evening,"Shinjuku (Tokyo Metropolitan Government Building, Kabukicho/Golden Gai)",Explore Shinjuku. Visit the Tokyo Metropolitan Government Building for free panoramic views (until late evening). Wander through the bustling streets of Kabukicho or the quaint Golden Gai for a unique nightlife experience.,The Tokyo Metropolitan Government Building offers great views without a fee. Golden Gai is a dense area of tiny bars.
Time,Location,Description,Notes
Morning,"Imperial Palace East Garden & Chidorigafuchi Moat, Chiyoda","Visit the serene Imperial Palace East Garden, part of the former Edo Castle grounds. If cherry blossoms are still in season, take a boat ride or walk along Chidorigafuchi Moat.",The East Garden is closed on Mondays and Fridays. Chidorigafuchi is a prime cherry blossom spot.
Afternoon,Ginza,"Explore Ginza, Tokyo's premier luxury shopping district. Admire the architecture, browse high-end boutiques, and perhaps visit a department store like Ginza Mitsukoshi or Ginza Wako.",Ginza is known for its upscale shopping and dining. Pedestrian paradise on weekends.
